In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from sklearn.metrics import confusion_matrix, cohen_kappa_score
import os


In [ ]:
LLM_IDENTIFIERS = ["Claude Opus 4.8", "Gemini 3.1 Pro", "GPT-5.5", "Llama 3.1 70B"]
CATEGORY_HIERARCHY_REACTIONS = {
    "Reaction feasible, all good": 5,  # Most positive
    "Reaction feasible, unexpected disconnection": 5,  
    "Protecting group strategy is wrong / non-optimal": 3,  
    "Non-optimal reagent": 4,
    "Unnecessary step":4,
    "Selectivity (regio-, stereo-, chemo-) issues": 2,
    "Problems with reaction type and functional group compatibility": 2,  
    "Unlikely disconnection": 1  # Most negative
}

# Sentiment categories
SENTIMENT_MAPPING_REACTIONS = {
    "Reaction feasible, all good": "Positive",
    "Reaction feasible, unexpected disconnection": "Positive", 
    "Non-optimal reagent": "Negative",
    "Unnecessary step":"Negative",
    "Protecting group strategy is wrong / non-optimal": "Negative",
    "Selectivity (regio-, stereo-, chemo-) issues": "Negative",
    "Problems with reaction type and functional group compatibility": "Negative",
    "Unlikely disconnection": "Negative"
}
aliases = {
    "Reaction feasible, all good": "Reaction feasible",
    "Reaction feasible, unexpected disconnection": "Unexpected disconnection", 
    "Non-optimal reagent": "Non-optimal reagent",
    "Unnecessary step":"Unnecessary step",
    "Protecting group strategy is wrong / non-optimal": "Protecting group issues",
    "Selectivity (regio-, stereo-, chemo-) issues": "Selectivity issues",
    "Problems with reaction type and functional group compatibility": "Functional group issues",
    "Unlikely disconnection": "Unlikely disconnection"
}
# Define category hierarchy (from most positive to most negative)
CATEGORY_HIERARCHY_ROUTES = {
    "Route feasible": 5,  # Most positive
    "Route feasible with few modifications": 4,  
    "Route feasible with significant modifications": 2,  
    "Route unfeasible": 1,
    "Route was not solved to building blocks":1,
}

# Define sentiment categories
SENTIMENT_MAPPING_ROUTES = {
    "Route feasible": "Positive",
    "Route feasible with few modifications": "Positive", 
    "Route feasible with significant modifications": "Negative",
    "Route unfeasible":"Negative",
    "Route was not solved to building blocks": "Negative",
}


In [ ]:
LLM_IDENTIFIERS = ["Claude Opus 4.8", "Gemini 3.1 Pro", "GPT-5.5", "Llama 3.1 70B"]

class CategoryMappings:
    """Container for category mappings and hierarchy for different analysis types"""
    
    REACTIONS = {
        'hierarchy': {
            "Reaction feasible, all good": 5,
            "Reaction feasible, unexpected disconnection": 5,
            "Non-optimal reagent": 4,
            "Unnecessary step": 4,
            "Protecting group strategy is wrong / non-optimal": 3,
            "Selectivity (regio-, stereo-, chemo-) issues": 2,
            "Problems with reaction type and functional group compatibility": 2,
            "Unlikely disconnection": 1
        },
        'sentiment': {
            "Reaction feasible, all good": "Positive",
            "Reaction feasible, unexpected disconnection": "Positive",
            "Non-optimal reagent": "Negative",
            "Unnecessary step": "Negative",
            "Protecting group strategy is wrong / non-optimal": "Negative",
            "Selectivity (regio-, stereo-, chemo-) issues": "Negative",
            "Problems with reaction type and functional group compatibility": "Negative",
            "Unlikely disconnection": "Negative"
        },
        'aliases': {
            "Reaction feasible, all good": "Reaction feasible",
            "Reaction feasible, unexpected disconnection": "Unexpected disconnection",
            "Non-optimal reagent": "Non-optimal reagent",
            "Unnecessary step": "Unnecessary step",
            "Protecting group strategy is wrong / non-optimal": "Protecting group issues",
            "Selectivity (regio-, stereo-, chemo-) issues": "Selectivity issues",
            "Problems with reaction type and functional group compatibility": "Functional group issues",
            "Unlikely disconnection": "Unlikely disconnection"
        }
    }
    
    ROUTES = {
        'hierarchy': {
            "Route feasible as it is": 5,
            "Route feasible with few modifications": 4,
            "Route feasible with significant modifications": 2,
            "Route unfeasible": 1,
            "Route was not solved to building blocks": 1,
        },
        'sentiment': {
            "Route feasible as it is": "Positive",
            "Route feasible with few modifications": "Positive",
            "Route feasible with significant modifications": "Negative",
            "Route unfeasible": "Negative",
            "Route was not solved to building blocks": "Negative",
        },
        'aliases': {
            "Route feasible as it is": "Feasible as it is",
            "Route feasible with few modifications": "Few modifications",
            "Route feasible with significant modifications": "Significant modifications",
            "Route unfeasible": "Unfeasible",
            "Route was not solved to building blocks": "Not solved",
        }
    }
    
    @classmethod
    def get_mappings(cls, hash_column):
        """Get the appropriate mappings based on hash_column"""
        if hash_column == 'reaction_hash':
            return cls.REACTIONS
        else:
            return cls.ROUTES


class ConsensusAnalyzer:
    """Handles consensus analysis with pessimistic tie-breaking"""
    
    def __init__(self, hash_column):
        self.mappings = CategoryMappings.get_mappings(hash_column)
        self.hierarchy = self.mappings['hierarchy']
        self.sentiment_mapping = self.mappings['sentiment']
    
    def _pick_pessimistic_category(self, categories):
        """Pick the most pessimistic category from a list"""
        if not categories:
            return None
        # Lower hierarchy value = worse (more pessimistic)
        # For categories missing from hierarchy, assign -inf so they sort first (most pessimistic)
        return sorted(categories, key=lambda c: self.hierarchy.get(c, -np.inf))[0]
    
    def get_majority_category_with_pessimism(self, categories):
        """
        Apply 3-step rule for consensus with pessimistic tie-breaking:
        1) If strict majority exists, choose it
        2) Else, compute sentiment majority; choose categories within that sentiment
           with max votes; if multiple tie, choose 'worse' per hierarchy
        3) If sentiment is tied, apply pessimistic tie-breaking across all tied categories
        
        Returns: (chosen_category, chosen_sentiment, votes_by_category, votes_by_sentiment)
        """
        if not categories:
            return "No feedback", "Unknown", Counter(), Counter()

        cat_counts = Counter(categories)
        total = sum(cat_counts.values())
        
        # Step 1: Check for strict majority
        top_cat, top_count = cat_counts.most_common(1)[0]
        if top_count > total / 2.0:
            sent_counts = Counter(self.sentiment_mapping.get(c, "Unknown") for c in categories)
            return top_cat, self.sentiment_mapping.get(top_cat, "Unknown"), cat_counts, sent_counts

        # Build sentiment counts
        sentiments = [self.sentiment_mapping.get(c, "Unknown") for c in categories]
        sent_counts = Counter(sentiments)

        # Step 2: Check for single sentiment (no sentiment competition)
        if len(sent_counts) == 1:
            max_count = max(cat_counts.values())
            tied = [c for c, v in cat_counts.items() if v == max_count]
            choice = self._pick_pessimistic_category(tied)
            return choice, self.sentiment_mapping.get(choice, "Unknown"), cat_counts, sent_counts

        # Check for sentiment majority (clear winner)
        sent_sorted = sent_counts.most_common()
        sent_top, sent_top_count = sent_sorted[0]
        sent_second_count = sent_sorted[1][1] if len(sent_sorted) > 1 else 0

        if sent_top_count != sent_second_count:
            # Sentiment majority exists - pick best category within winning sentiment
            within_sentiment = {c: v for c, v in cat_counts.items() 
                              if self.sentiment_mapping.get(c, "Unknown") == sent_top}
            max_within = max(within_sentiment.values())
            tied_within = [c for c, v in within_sentiment.items() if v == max_within]
            choice = self._pick_pessimistic_category(tied_within)
            return choice, sent_top, cat_counts, sent_counts

        # Step 3: Sentiment tie - pessimistic across all categories tied at max count
        max_count = max(cat_counts.values())
        tied_all = [c for c, v in cat_counts.items() if v == max_count]
        choice = self._pick_pessimistic_category(tied_all)
        return choice, self.sentiment_mapping.get(choice, "Unknown"), cat_counts, sent_counts


class GroupConsensusAnalyzer:
    """Handles group consensus analysis for humans vs LLMs"""
    
    def __init__(self, hash_column):
        self.consensus_analyzer = ConsensusAnalyzer(hash_column)
        self.mappings = CategoryMappings.get_mappings(hash_column)
    
    def get_group_consensus(self, group_data, expert_column, category_column, 
                          text_column=None, confidence_column=None):
        """
        Get consensus information for a group (human or single LLM) on a single reaction
        """
        experts = group_data[expert_column].unique()
        num_experts = len(experts)

        # Detect LLM group (single LLM name) vs humans
        is_llm = (len(experts) == 1) and (experts[0] in LLM_IDENTIFIERS)

        if is_llm:
            # LLM: votes are all rows' categories (re-runs)
            votes = group_data[category_column].dropna().tolist()
            num_voters = len(votes)
            
            if num_voters == 0:
                return self._empty_consensus_result()
            
            dominant_category, dominant_sentiment, cat_counts, _ = \
                self.consensus_analyzer.get_majority_category_with_pessimism(votes)
            
            agreement_pct = (cat_counts.get(dominant_category, 0) / num_voters * 100.0)
            category_breakdown = {
                c: {'count': cnt, 'percentage': (cnt / num_voters * 100.0)} 
                for c, cnt in cat_counts.items()
            }
            
            all_text_feedback = []
            if text_column and text_column in group_data.columns:
                all_text_feedback = group_data[text_column].dropna().tolist()
            
            return {
                'dominant_category': dominant_category,
                'dominant_sentiment': dominant_sentiment,
                'agreement_pct': agreement_pct,
                'num_experts': num_voters,  # number of votes (re-runs)
                'category_breakdown': category_breakdown,
                'text_feedback': all_text_feedback,
                'all_categories': dict(cat_counts)
            }
        
        else:
            # Humans: PRESENCE VOTING (each expert contributes at most one vote per category)
            category_counts = defaultdict(int)
            all_text_feedback = []
            
            for expert in experts:
                expert_rows = group_data[group_data[expert_column] == expert]
                
                # Each category counted once per expert (presence vote)
                expert_categories = set(expert_rows[category_column].dropna().tolist())
                for cat in expert_categories:
                    category_counts[cat] += 1
                
                # Optional text aggregation
                if text_column and text_column in expert_rows.columns:
                    expert_texts = expert_rows[text_column].dropna().tolist()
                    for txt in expert_texts:
                        if isinstance(txt, str) and txt.strip():
                            all_text_feedback.append({'expert': expert, 'text': txt.strip()})
            
            if not category_counts:
                return self._empty_consensus_result()
            
            # Build expanded vote list from presence counts and resolve with pessimistic tie-breaking
            expanded_votes = [cat for cat, count in category_counts.items() for _ in range(count)]
            dominant_category, dominant_sentiment, _, _ = \
                self.consensus_analyzer.get_majority_category_with_pessimism(expanded_votes)
            
            # Agreement percentage over number of experts
            agreement_pct = (category_counts.get(dominant_category, 0) / num_experts * 100.0)
            
            # Breakdown (per expert basis)
            category_breakdown = {
                c: {'count': count, 'percentage': (count / num_experts * 100.0)}
                for c, count in category_counts.items()
            }
            
            return {
                'dominant_category': dominant_category,
                'dominant_sentiment': dominant_sentiment,
                'agreement_pct': agreement_pct,
                'num_experts': num_experts,  # unique humans contributing
                'category_breakdown': category_breakdown,
                'text_feedback': all_text_feedback,
                'all_categories': dict(category_counts)
            }
    
    def _empty_consensus_result(self):
        """Return empty consensus result structure"""
        return {
            'dominant_category': "No feedback",
            'dominant_sentiment': "Unknown",
            'agreement_pct': 0.0,
            'num_experts': 0,
            'category_breakdown': {},
            'text_feedback': [],
            'all_categories': {}
        }


class HumanLLMComparator:
    """Handles comparison between human and LLM evaluations"""
    
    def __init__(self, hash_column):
        self.mappings = CategoryMappings.get_mappings(hash_column)
        self.hierarchy = self.mappings['hierarchy']
        self.sentiment_mapping = self.mappings['sentiment']
    
    def compare_group_evaluations(self, human_analysis, llm_analysis):
        """Compare human vs LLM consensus with sentiment and score analysis"""
        human_category = human_analysis['dominant_category']
        llm_category = llm_analysis['dominant_category']

        human_sentiment = self.sentiment_mapping.get(human_category, 'Unknown')
        llm_sentiment = self.sentiment_mapping.get(llm_category, 'Unknown')

        human_score = self.hierarchy.get(human_category, 0)
        llm_score = self.hierarchy.get(llm_category, 0)

        sentiment_agreement = human_sentiment == llm_sentiment
        category_agreement = human_category == llm_category
        score_difference = abs(human_score - llm_score)

        if category_agreement:
            disagreement_severity = "No disagreement"
        elif sentiment_agreement:
            disagreement_severity = "Mild disagreement (same sentiment)"
        elif score_difference <= 1:
            disagreement_severity = "Moderate disagreement"
        else:
            disagreement_severity = "Strong disagreement"

        return {
            'sentiment_agreement': sentiment_agreement,
            'category_agreement': category_agreement,
            'score_difference': score_difference,
            'disagreement_severity': disagreement_severity,
            'human_sentiment': human_sentiment,
            'llm_sentiment': llm_sentiment,
            'human_score': human_score,
            'llm_score': llm_score
        }


In [ ]:
def generate_confusion_matrices(results_df):
    """Generate confusion matrices for different levels of comparison"""
    # Category-level confusion matrix
    human_categories = results_df['human_dominant_category'].tolist()
    llm_categories = results_df['llm_dominant_category'].tolist()
    
    # Get all unique categories
    all_categories = sorted(list(set(human_categories + llm_categories)))
    
    # Create confusion matrix
    cm_categories = confusion_matrix(human_categories, llm_categories, labels=all_categories)
    
    # Sentiment-level confusion matrix
    human_sentiments = results_df['human_sentiment'].tolist()
    llm_sentiments = results_df['llm_sentiment'].tolist()
    
    sentiment_labels = ['Positive', 'Negative']
    cm_sentiments = confusion_matrix(human_sentiments, llm_sentiments, labels=sentiment_labels)
    
    # Score-level data
    human_scores = results_df['human_score'].tolist()
    llm_scores = results_df['llm_score'].tolist()
    
    return {
        'category_matrix': {
            'matrix': cm_categories,
            'labels': all_categories
        },
        'sentiment_matrix': {
            'matrix': cm_sentiments,
            'labels': sentiment_labels
        },
        'human_categories': human_categories,
        'llm_categories': llm_categories,
        'human_scores': human_scores,
        'llm_scores': llm_scores
    }


def calculate_agreement_statistics(results_df):
    """Calculate various agreement statistics"""
    total_reactions = len(results_df)
    
    if total_reactions == 0:
        return {
            'total_reactions': 0,
            'category_agreement_count': 0,
            'category_agreement_percentage': 0.0,
            'sentiment_agreement_count': 0,
            'sentiment_agreement_percentage': 0.0,
            'disagreement_distribution': {},
            'disagreement_distribution_percentage': {},
            'cohens_kappa': 0.0,
            'mean_score_difference': 0.0,
            'std_score_difference': 0.0,
            'max_score_difference': 0
        }
    
    # Category agreement
    category_agreement = results_df['category_agreement'].sum()
    category_agreement_pct = (category_agreement / total_reactions) * 100
    
    # Sentiment agreement
    sentiment_agreement = results_df['sentiment_agreement'].sum()
    sentiment_agreement_pct = (sentiment_agreement / total_reactions) * 100
    
    # Disagreement severity distribution
    disagreement_dist = results_df['disagreement_severity'].value_counts()
    disagreement_dist_pct = (disagreement_dist / total_reactions * 100).round(1)
    
    # Cohen's Kappa for categories
    human_categories = results_df['human_dominant_category'].tolist()
    llm_categories = results_df['llm_dominant_category'].tolist()
    
    # Convert categories to numeric for kappa calculation
    all_categories = sorted(set(human_categories + llm_categories))
    if len(all_categories) > 1:
        category_to_num = {cat: i for i, cat in enumerate(all_categories)}
        human_numeric = [category_to_num[cat] for cat in human_categories]
        llm_numeric = [category_to_num[cat] for cat in llm_categories]
        kappa_score = cohen_kappa_score(human_numeric, llm_numeric)
    else:
        kappa_score = 1.0  # Perfect agreement if only one category
    
    # Score difference statistics
    score_differences = results_df['score_difference']
    
    return {
        'total_reactions': total_reactions,
        'category_agreement_count': int(category_agreement),
        'category_agreement_percentage': round(category_agreement_pct, 1),
        'sentiment_agreement_count': int(sentiment_agreement),
        'sentiment_agreement_percentage': round(sentiment_agreement_pct, 1),
        'disagreement_distribution': disagreement_dist.to_dict(),
        'disagreement_distribution_percentage': disagreement_dist_pct.to_dict(),
        'cohens_kappa': round(kappa_score, 3),
        'mean_score_difference': round(score_differences.mean(), 2),
        'std_score_difference': round(score_differences.std(), 2),
        'max_score_difference': int(score_differences.max()) if len(score_differences) > 0 else 0
    }


def analyze_disagreements(results_df):
    """Analyze patterns in disagreements"""
    # Filter for disagreements
    disagreements = results_df[~results_df['category_agreement']].copy()
    
    if disagreements.empty:
        return {'message': 'No disagreements found'}
    
    # Most common disagreement patterns
    disagreement_patterns = []
    for _, row in disagreements.iterrows():
        pattern = f"Human: {row['human_dominant_category']} → LLM: {row['llm_dominant_category']}"
        disagreement_patterns.append(pattern)
    
    pattern_counts = Counter(disagreement_patterns)
    
    # Reactions with strongest disagreements
    strongest_disagreements = disagreements.nlargest(10, 'score_difference')
    
    # Sentiment flip analysis
    sentiment_flips = disagreements[~disagreements['sentiment_agreement']]
    
    return {
        'total_disagreements': len(disagreements),
        'disagreement_percentage': round((len(disagreements) / len(results_df)) * 100, 1),
        'most_common_patterns': dict(pattern_counts.most_common(10)),
        'strongest_disagreements': strongest_disagreements[[
            'reaction_hash', 'human_dominant_category', 'llm_dominant_category',  
            'score_difference', 'disagreement_severity'
        ]].to_dict('records'),
        'sentiment_flips': len(sentiment_flips),
        'sentiment_flip_percentage': round((len(sentiment_flips) / len(disagreements)) * 100, 1) if len(disagreements) > 0 else 0
    }


def compare_category_distributions(human_feedback, llm_feedback, category_column, expert_column):
    """Compare overall category distributions between humans and LLMs"""
    # Get category distributions
    human_categories = human_feedback[category_column].value_counts()
    llm_categories = llm_feedback[category_column].value_counts()
    
    # Normalize to percentages
    human_pct = (human_categories / human_categories.sum() * 100).round(1) if len(human_categories) > 0 else pd.Series()
    llm_pct = (llm_categories / llm_categories.sum() * 100).round(1) if len(llm_categories) > 0 else pd.Series()
    
    # Combine into comparison DataFrame
    all_categories = sorted(set(human_categories.index.tolist() + llm_categories.index.tolist()))
    
    comparison_data = []
    for category in all_categories:
        comparison_data.append({
            'category': category,
            'human_count': human_categories.get(category, 0),
            'human_percentage': human_pct.get(category, 0),
            'llm_count': llm_categories.get(category, 0),
            'llm_percentage': llm_pct.get(category, 0),
            'difference': human_pct.get(category, 0) - llm_pct.get(category, 0)
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    comparison_df = comparison_df.sort_values('difference', key=abs, ascending=False)
    
    return comparison_df


def analyze_human_vs_llm_agreement(feedback_df, hash_column='reaction_hash',
                                 category_column='local_feedback',
                                 expert_column='source_file',
                                 text_column='local_feedback_text',
                                 confidence_column='confidence',
                                 llm_repeat=0):
    """
    Comprehensive analysis comparing human majority vs EACH LLM majority,
    using pessimistic tie-breaking for consensus selection on both sides.
    """
    # Initialize analyzers based on hash_column
    group_analyzer = GroupConsensusAnalyzer(hash_column)
    comparator = HumanLLMComparator(hash_column)
    
    # Separate human and LLM feedback
    feedback_df = feedback_df.dropna(subset=[hash_column])
    
    human_feedback = feedback_df[~feedback_df[expert_column].isin(LLM_IDENTIFIERS)].copy()
    llm_feedback = feedback_df[feedback_df[expert_column].isin(LLM_IDENTIFIERS)].copy()
    # Use one repeat so LLMs have the same vote count as one independent run
    # (4 reruns), symmetric with the human expert pool.
    if llm_repeat is not None and 'repeat' in llm_feedback.columns:
        llm_feedback = llm_feedback[llm_feedback['repeat'] == llm_repeat].copy()

    print(f"Human experts: {len(human_feedback[expert_column].unique())}")
    print(f"LLM experts: {len(llm_feedback[expert_column].unique())}")
    print(f"Total human feedback entries: {len(human_feedback)}")
    print(f"Total LLM feedback entries: {len(llm_feedback)}")

    # Reactions overlap
    human_reactions = set(human_feedback[hash_column].unique())
    llm_reactions = set(llm_feedback[hash_column].unique())
    common_reactions = human_reactions.intersection(llm_reactions)

    print(f"Reactions evaluated by humans only: {len(human_reactions - llm_reactions)}")
    print(f"Reactions evaluated by LLMs only: {len(llm_reactions - human_reactions)}")
    print(f"Reactions evaluated by both: {len(common_reactions)}")

    if not common_reactions:
        print("No reactions found that were evaluated by both humans and LLMs.")
        return {}

    # Precompute human consensus per reaction once
    human_consensus_by_rxn = {}
    for rxn in common_reactions:
        hdata = human_feedback[human_feedback[hash_column] == rxn]
        human_consensus_by_rxn[rxn] = group_analyzer.get_group_consensus(
            hdata, expert_column, category_column, text_column, confidence_column
        )

    # Analyze for each LLM separately
    comparison_results = []
    for llm_name in LLM_IDENTIFIERS:
        llm_df = llm_feedback[llm_feedback[expert_column] == llm_name]
        llm_rxns = set(llm_df[hash_column].unique())
        rxns_to_compare = sorted(common_reactions.intersection(llm_rxns))

        for reaction_hash in rxns_to_compare:
            # Human consensus (precomputed)
            human_analysis = human_consensus_by_rxn[reaction_hash]

            # LLM consensus (re-runs for this LLM on this reaction)
            llm_data = llm_df[llm_df[hash_column] == reaction_hash]
            llm_analysis = group_analyzer.get_group_consensus(
                llm_data, expert_column, category_column, text_column, confidence_column
            )

            # Compare majority human vs majority LLM
            comparison = comparator.compare_group_evaluations(human_analysis, llm_analysis)

            result = {
                'llm_name': llm_name,
                'reaction_hash': reaction_hash,
                'human_dominant_category': human_analysis['dominant_category'],
                'human_agreement_pct': human_analysis['agreement_pct'],
                'human_sentiment': comparison['human_sentiment'],
                'human_score': comparison['human_score'],
                'human_experts': human_analysis['num_experts'],
                'llm_dominant_category': llm_analysis['dominant_category'],
                'llm_agreement_pct': llm_analysis['agreement_pct'],
                'llm_sentiment': comparison['llm_sentiment'],
                'llm_score': comparison['llm_score'],
                'llm_num_runs': llm_analysis['num_experts'],
                'sentiment_agreement': comparison['sentiment_agreement'],
                'category_agreement': comparison['category_agreement'],
                'score_difference': comparison['score_difference'],
                'disagreement_severity': comparison['disagreement_severity'],
                'human_category_breakdown': human_analysis['category_breakdown'],
                'llm_category_breakdown': llm_analysis['category_breakdown']
            }

            comparison_results.append(result)

    # Create DataFrame
    results_df = pd.DataFrame(comparison_results)

    # Build per-LLM sentiment confusion counts
    matrix_rows = []
    for llm_name in LLM_IDENTIFIERS:
        sub = results_df[results_df['llm_name'] == llm_name]
        if len(sub) > 0:
            TP = int(((sub['human_sentiment'] == 'Positive') & (sub['llm_sentiment'] == 'Positive')).sum())
            TN = int(((sub['human_sentiment'] == 'Negative') & (sub['llm_sentiment'] == 'Negative')).sum())
            FP = int(((sub['human_sentiment'] == 'Negative') & (sub['llm_sentiment'] == 'Positive')).sum())
            FN = int(((sub['human_sentiment'] == 'Positive') & (sub['llm_sentiment'] == 'Negative')).sum())
        else:
            TP = TN = FP = FN = 0
        matrix_rows.append({'llm_name': llm_name, 'TN': TN, 'FN': FN, 'FP': FP, 'TP': TP})
    matrix_df = pd.DataFrame(matrix_rows)

    # Calculate agreement statistics
    agreement_statistics = calculate_agreement_statistics(results_df)

    # Category distribution comparison
    category_comparison = compare_category_distributions(
        human_feedback, llm_feedback, category_column, expert_column
    )

    # Confusion matrices
    confusion_matrix_bundle = generate_confusion_matrices(results_df)

    analysis = {
        'results_df': results_df,
        'matrix_df': matrix_df,
        'confusion_matrix': confusion_matrix_bundle,
        'agreement_statistics': agreement_statistics,
        'disagreement_analysis': analyze_disagreements(results_df),
        'category_comparison': category_comparison
    }
    
    return analysis


def run_human_vs_llm_analysis(feedback_csv, hash_column='reaction_hash', 
                             output_dir='human_vs_llm_analysis',
                             category_column='local_feedback',
                             expert_column='source_file',
                             text_column='local_feedback_text',
                             confidence_column='confidence',
                             llm_repeat=0):
    """
    Run complete human vs LLM analysis with specified hash_column
    """
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Load data
    feedback_df = pd.read_csv(feedback_csv)
    
    print("Starting Human vs LLM Agreement Analysis...")
    print("=" * 50)
    print(f"Analysis type: {'Reactions' if hash_column == 'reaction_hash' else 'Routes'}")
    
    # Run analysis
    analysis_results = analyze_human_vs_llm_agreement(
        feedback_df, 
        hash_column=hash_column,
        category_column=category_column,
        expert_column=expert_column,
        text_column=text_column,
        confidence_column=confidence_column,
        llm_repeat=llm_repeat
    )
    
    if not analysis_results:
        print("Analysis failed. Please check your data.")
        return
    
    # Save results
    results_df = analysis_results['results_df']
    results_df.to_csv(os.path.join(output_dir, 'human_vs_llm_comparison.csv'), index=False)
    
    # Save disagreement details
    disagreement_analysis = analysis_results['disagreement_analysis']
    if 'strongest_disagreements' in disagreement_analysis:
        disagreement_df = pd.DataFrame(disagreement_analysis['strongest_disagreements'])
        disagreement_df.to_csv(os.path.join(output_dir, 'strongest_disagreements.csv'), index=False)
    
    # Save category comparison
    category_comparison = analysis_results['category_comparison']
    category_comparison.to_csv(os.path.join(output_dir, 'category_distribution_comparison.csv'), index=False)
    
    print(f"\nAnalysis complete! Results saved to {output_dir}/")
    
    return analysis_results